# Фінальний пайплайн: інференс RT-DETR на більшій роздільній здатності (без перетренування)
Q0/Q1 натреновані на 1024 px. Тут ті самі ваги проганяються на 1024 і 1280 px: на val перевіряємо, чи більша картинка
допомагає дрібним об'єктам; найкращу пару (модель, imgsz) застосовуємо до тесту.
Input: дані змагання + Output ноутбука RT-DETR (Q0, Q1). GPU: одна T4 або P100.

In [1]:
import importlib.util, subprocess, sys
for pkg in ["ultralytics", "pycocotools"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import contextlib, gc, io, logging, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)

MAX_DET, CONF = 500, 0.001
ROOT = next(p.parent.parent for p in Path("/kaggle/input").rglob("val/annotations_coco.json"))
WEIGHTS = {p.parent.name: p for p in Path("/kaggle/input").rglob("best.pt")
           if p.parent.name in ("Q0_rtdetr_l_1024", "Q1_rtdetr_l_nq500_1024")}
OUT = Path("/kaggle/working")
print("дані:", ROOT, "\nваги:", list(WEIGHTS))


def frame_key(path):
    p = Path(path)
    return p.parent.name, p.stem


clips = sorted(p for p in (ROOT / "val" / "images").iterdir() if p.is_dir())
VAL = [f for c in clips for f in sorted(c.glob("*.jpg"))[::2]]
test_df = pd.read_csv(ROOT / "test_images.csv")
TEST = [ROOT / f for f in test_df.file_name]
with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(str(ROOT / "val" / "annotations_coco.json"))
KEY2ID = {frame_key(i["file_name"]): i["id"] for i in GT.dataset["images"]}


def predict(model, paths, imgsz, batch=8, chunk=64):
    preds = {}
    paths = [str(p) for p in paths]
    for s in range(0, len(paths), chunk):
        for r in model.predict(source=paths[s:s + chunk], imgsz=imgsz, conf=CONF, max_det=MAX_DET, half=True,
                               batch=batch, stream=True, verbose=False, device=0):
            b = r.boxes
            preds[frame_key(r.path)] = (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int))
        gc.collect(); torch.cuda.empty_cache()
    return preds


def evaluate(preds):
    dets = []
    for k, (b, s, c) in preds.items():
        for i in np.argsort(-s)[:MAX_DET]:
            x1, y1, x2, y2 = map(float, b[i])
            dets.append(dict(image_id=KEY2ID[k], category_id=int(c[i]) + 1, bbox=[x1, y1, x2 - x1, y2 - y1], score=float(s[i])))
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(GT, GT.loadRes(dets), "bbox")
        E.params.imgIds = [KEY2ID[k] for k in preds]
        E.params.maxDets = [1, 100, MAX_DET]
        E.evaluate(); E.accumulate(); E.summarize()
    p = E.eval["precision"][:, :, :, 0, -1]
    return dict(mAP=float(p[p > -1].mean()), AP50=float(E.stats[1]), AP_small=float(E.stats[3]),
                AP_medium=float(E.stats[4]), AP_large=float(E.stats[5]))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
дані: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 
ваги: ['Q0_rtdetr_l_1024', 'Q1_rtdetr_l_nq500_1024']


## Val: модель × роздільна здатність інференсу

In [2]:
rows = []
for name, w in WEIGHTS.items():
    model = RTDETR(str(w))
    for imgsz in [1024, 1280]:
        t0 = time.time()
        m = evaluate(predict(model, VAL, imgsz))
        rows.append(dict(model=name, imgsz=imgsz, **m))
        print(f"{name} @ {imgsz}: mAP {m['mAP']:.4f} ({time.time() - t0:.0f} с)")
    del model; gc.collect(); torch.cuda.empty_cache()
res = pd.DataFrame(rows).sort_values("mAP", ascending=False)
display(res.round(4))
res.to_csv(OUT / "rtdetr_infer_size.csv", index=False)

Q0_rtdetr_l_1024 @ 1024: mAP 0.2683 (271 с)
Q0_rtdetr_l_1024 @ 1280: mAP 0.2816 (271 с)
Q1_rtdetr_l_nq500_1024 @ 1024: mAP 0.2613 (287 с)
Q1_rtdetr_l_nq500_1024 @ 1280: mAP 0.2706 (330 с)


,model,imgsz,mAP,AP50,AP_small,AP_medium,AP_large
1,Q0_rtdetr_l_1024,1280,0.2816,0.5300,0.1500,0.2439,0.4677
3,Q1_rtdetr_l_nq500_1024,1280,0.2706,0.5114,0.1378,0.2359,0.4535
0,Q0_rtdetr_l_1024,1024,0.2683,0.5135,0.1389,0.2310,0.4562
2,Q1_rtdetr_l_nq500_1024,1024,0.2613,0.4997,0.1245,0.2255,0.4371


## Тест: найкраща пара (модель, imgsz) з тих, що відрізняються від уже засабмічених 1024-версій

In [3]:
cand = res[res.imgsz != 1024]
best = cand.iloc[0]
base = res[(res.model == best.model) & (res.imgsz == 1024)].iloc[0]
print(f"{best.model} @ {best.imgsz}: mAP {best.mAP:.4f} vs @1024: {base.mAP:.4f} ({(best.mAP - base.mAP) * 100:+.2f} п.п.)")

model = RTDETR(str(WEIGHTS[best.model]))
preds = predict(model, TEST, int(best.imgsz))
rows = []
for iid, fn in zip(test_df.image_id, test_df.file_name):
    b, s, c = preds[frame_key(fn)]
    w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
    keep = np.where((w >= 1) & (h >= 1))[0]
    o = keep[np.argsort(-s[keep])][:MAX_DET]
    rows.append((iid, " ".join(f"{int(c[i])} {s[i]:.4f} {b[i, 0]:.1f} {b[i, 1]:.1f} {w[i]:.1f} {h[i]:.1f}" for i in o)))
path = OUT / f"submission_{best.model}_infer{int(best.imgsz)}.csv"
pd.DataFrame(rows, columns=["image_id", "PredictionString"]).to_csv(path, index=False)
print("сабміт:", path)

Q0_rtdetr_l_1024 @ 1280: mAP 0.2816 vs @1024: 0.2683 (+1.32 п.п.)
сабміт: /kaggle/working/submission_Q0_rtdetr_l_1024_infer1280.csv
